# RAG pipeline

In [1]:
import os
import sys
import csv
import ast
import json
import asyncio
import regex as re
import numpy as np
import pandas as pd
from ast import literal_eval
from dotenv import load_dotenv
from llama_index.core import (
    Document,
    VectorStoreIndex,
    StorageContext,
    PromptTemplate,
    load_index_from_storage,
    set_global_handler
)
from llama_index.llms.vllm import Vllm
from llama_index.llms.groq import Groq
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.settings import Settings
from llama_index.core.llms import ChatMessage
from llama_cloud import AsyncLlamaCloud

# logging
import logging
logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("llama_index.core").setLevel(logging.WARNING)
logging.getLogger("fsspec").setLevel(logging.WARNING)
set_global_handler("simple")

CACHE_DIR = "./parsed_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

load_dotenv()
client = AsyncLlamaCloud(api_key=os.getenv("LLAMA_CLOUD_API_KEY"))
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

eval_dataset = pd.read_csv("./evaluation/eval_dataset.csv")
correctness_prompt = open("./evaluation/prompts/correctness.md").read()
relevance_prompt = open("./evaluation/prompts/relevance.md").read()
faithfulness_prompt = open("./evaluation/prompts/faithfulness.md").read()

## Setup LLM and Embedding model

* For the LLM, use **Gemini** for local development and *vLLM* for production
* For the embedding model use **BAAI/bge-small-en-v1.5** for local and Qwen3

In [2]:
if os.getenv("APP_ENV") == "dev":
    embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
    llm = Groq(model="qwen/qwen3-32b", api_key=GROQ_API_KEY)
elif os.getenv("APP_ENV") == "prod":
    embed_model = HuggingFaceEmbedding(model_name="Qwen/Qwen3-Embedding-0.6B")
    llm = Vllm(
        model="Qwen3.5-122B-A10B-GGUF",
        tensor_parallel_size=4,
        max_new_tokens=100,
        vllm_kwargs={"swap_space": 1, "gpu_memory_utilization": 0.5},
    )

Settings.llm = llm
Settings.embed_model = embed_model

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5


INFO:sentence_transformers.SentenceTransformer:1 prompt is loaded, with the key: query


## Parse the PDF

In [3]:
async def parse_documents_with_llamaparse(data_dir: str):
    documents = []

    for filename in os.listdir(data_dir):
        if not filename.endswith(".pdf"):
            continue

        cache_file = os.path.join(CACHE_DIR, f"{filename}.json")

        # load from cache
        if os.path.exists(cache_file):
            print(f"Loading cached parse for {filename}...")
            with open(cache_file, "r") as f:
                pages = json.load(f)

            for page in pages:
                documents.append(
                    Document(
                        text=page["text"],
                        metadata=page["metadata"]
                    )
                )
            continue

        # parse normally
        print(f"No cache found for {cache_file}. Parsing with LlamaCloud...")
        file_path = os.path.join(data_dir, filename)

        print(f"Uploading {filename} to LlamaCloud...")
        file_obj = await client.files.create(
            file=file_path,
            purpose="parse"
        )

        print("Parsing file...")
        result = await client.parsing.parse(
            file_id=file_obj.id,
            tier="cost_effective",
            version="latest",
            agentic_options={
                "custom_prompt": "This is an equipment manual..."
            },
            output_options={
                "markdown": {
                    "tables": {"output_tables_as_markdown": True},
                },
                "images_to_save": ["embedded"],
            },
            expand=["markdown"]
        )

        print("Saving to cache...")
        pages_to_save = []
        for page in result.markdown.pages:
            pages_to_save.append({
                "text": page.markdown,
                "metadata": {
                    "file_name": filename,
                    "page": page.page_number
                }
            })

            documents.append(
                Document(
                    text=page.markdown,
                    metadata={
                        "file_name": filename,
                        "page": page.page_number
                    }
                )
            )

        with open(cache_file, "w") as f:
            json.dump(pages_to_save, f)

    return documents

documents = await parse_documents_with_llamaparse("./data")

Loading cached parse for Getting_Started_with_SM_Datum_V7.2.4_2024_CT_EN.pdf...
Loading cached parse for User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN.pdf...


## Chunk the document and store the RAG index

NOTE: If the parsing or chunking strategy is changed it will NOT automatically update your existing stored index. It loads the old index with old chunks
* After applying a new parsing or chunking strategy, delete the old storage folder.

In [5]:
# vector index
if os.path.exists("./storage"):
    print("Loading index from storage...")
    storage_context = StorageContext.from_defaults(persist_dir="./storage")
    index = load_index_from_storage(storage_context)
else:
    splitter = SentenceSplitter(
        chunk_size=512,
        chunk_overlap=50
    )

    print("Chunking documents into nodes...")
    nodes = splitter.get_nodes_from_documents(documents)

    print("Creating new index...")
    index = VectorStoreIndex.from_documents(nodes)
    index.storage_context.persist("./storage")

Loading index from storage...


In [6]:
for node in nodes:
    print(f"{node.node_id}: {node.text}")

9b765a70-9365-400a-98b7-0b4b8eb862ef: www.contrastech.com

# Getting Started with SM-Datum

![Icon of a camera lens](page_1_image_1_v2.jpg)

Ver 2.3.4 Mar. 2024

Vision System
316fa863-4bc5-48fb-8655-f02deeeafd19: ![Perfect](page_2_image_1_v2.jpg)

# Perfect

## Purpose of This Guide

This guide describes the function of the SM-Datum and how to use it and gives a detailed description of each tools. The information contained in the Manual is subject to change, without notice, due to firmware updates or other reasons. Please find the latest version of this Manual at the Contrastech website.Please read this guide before use.

Copyright ©2023
Hangzhou Contrastech Co., Ltd.
Tel: 86-571-89712238
Add.: No.8, Xiyuan 9th Road West Lake District, Hangzhou 310030 China

All rights reserved. The information contained herein is proprietary and is provided solely for the purpose of allowing customers to operate and/or service Contrastech manufactured equipment and is not to be released, reproduced, 

## Retrieve Chunks and Generate Response

In [ ]:
# basic retriever
retriever = index.as_retriever(similarity_top_k=5)

print("Querying the index...")
query_engine = RetrieverQueryEngine.from_args(
    retriever,
    llm=llm
)

print("Generating response...")
response = await query_engine.aquery(
    "What is the name of this device?"
)

Querying the index...
Generating response...
** Messages: **
system: You are an expert Q&A system that is trusted around the world.
Always answer the query using the provided context information, and not prior knowledge.
Some rules to follow:
1. Never directly reference the given context in your answer.
2. Avoid statements like 'Based on the context, ...' or 'The context information ...' or anything along those lines.
user: Context information is below.
---------------------
file_name: User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN.pdf
page: 6

Equipment shown in Fig. 1-2

Double-click the SM-Datum shortcut on the desktop to open up the client software.

| Color  | Pin | Signal    | Signal Source | Designation                                                                    | Cable type               |
| ------ | --- | --------- | ------------- | ------------------------------------------------------------------------------ | ------------------------ |
| Red    | 1   | POWER\_IN | -      

In [ ]:
llm_response = response.response
print(llm_response)

<think>
Okay, let's see. The user is asking for the name of the device based on the provided context. Looking through the PDF pages, there are mentions of "SM2 SE Camera CT" in the file names. For example, the file name is "User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN.pdf". The pages mention things like the PWR, LNK, and STS indicators, which are status LEDs. There's also information about the power supply, I/O ports, and contact details for Hangzhou ContrasTech Co., Ltd. The device seems to be a camera with specific model numbers and technical specifications. The key part in the file name is "SM2 SE Camera CT", which likely stands for the model name. The version number is 2.5.4, but the user is asking for the device's name, not the version. The company name is Hangzhou ContrasTech, but the device itself is referred to as SM2 SE Camera CT. So the answer should be that the device is the SM2 SE Camera CT.
</think>

The device is referred to as the **SM2 SE Camera CT** in the provided docume

In [ ]:
# # add a new question to the csv
# with open("../eval_dataset.csv", "a", newline="", encoding="utf-8") as csvfile:
#     writer = csv.writer(csvfile)
#     writer.writerow([
#         "",
#         "",
#         [""]
#     ])

In [9]:
eval_dataset

,document,question,ground_truth,contexts
0,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,What is the order model number for the M12 cable?,['The order model number for the 17-pin M12 ca...,['### 17-pin M12 Cable with Fast Ethernet Inte...
1,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,What are the dimensions of the M12-mount housi...,['The dimensions of the M12-mount housing vari...,['Fig. 1-1: $46 \times 57.6 \times 25$ mm Mech...
2,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,What are all the pins on the 17-pin connector ...,['The pins on the 17-pin connector that should...,['| Purple White | 3 | - | - ...
3,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,What communication protocols does the SM2 seri...,['The SM2 series smart cameras supports Serial...,"['* Support Serial, TCP, UDP, FTP, Modbus and ..."
4,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,List all the LED indicators and what each colo...,"[""The LED indicators and their meanings are as...","[""| PWR Indicator | It is a power indicator. T..."
5,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,What are the step-by-step instructions to inst...,[''],['']
6,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,How do you add a remote camera in SM-Datum?,[''],['']
7,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,How does the wiring differ when connecting a P...,[''],['']
8,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,I want to use GPIO0 as an output. What needs t...,[''],['']
9,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,I want to connect 8 NPN photoelectric sensors ...,[''],['']


# Evaluation Framework

## Load dataset and generate responses

In [10]:
# convert stringified lists into real Python lists
eval_dataset["contexts"] = eval_dataset["contexts"].apply(ast.literal_eval)

# remove rows where contexts is empty or only contains empty strings
eval_dataset = eval_dataset[
    eval_dataset["contexts"].apply(
        lambda x: isinstance(x, list) and any(str(item).strip() for item in x)
    )
]

In [ ]:
async def run_pipeline():
    generated_answers = []
    retrieved_contexts = []

    for _, row in eval_dataset.iterrows():
        response = await query_engine.aquery(row["question"])
        generated_answers.append(response.response)
        retrieved_contexts.append([n.get_content() for n in response.source_nodes])
    
    return generated_answers, retrieved_contexts

generated_answers, retrieved_contexts = await run_pipeline()

## Evaluate Retrieval

In [11]:
def normalize(text: str) -> str:
    """Light normalization: collapse whitespace, lowercase."""
    import re
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)  # collapse newlines, tabs, extra spaces
    return text.strip()

def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    a, b = np.array(vec_a), np.array(vec_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

def is_relevant(
    retrieved_chunk: str,
    gt_chunks: list[str],
    embed_model,
    threshold: float
) -> bool:
    """
    A retrieved chunk is relevant if its cosine similarity to ANY
    ground truth chunk exceeds the threshold.
    """
    retrieved_vec = embed_model.get_text_embedding(normalize(retrieved_chunk))
    for gt_chunk in gt_chunks:
        gt_vec = embed_model.get_text_embedding(normalize(gt_chunk))
        if cosine_similarity(retrieved_vec, gt_vec) >= threshold:
            return True
    return False

In [ ]:
def evaluate_retrieval(
    query_engine,
    eval_dataset: pd.DataFrame,
    embed_model,
    threshold: float = 0.80,
    k: int = 5,
) -> dict:
    """
    Evaluate retrieval using MRR@K and Recall@K.
    """
    results = []

    for _, row in eval_dataset.iterrows():
        question = row["question"]

        # contexts column may be stored as a string representation of a list
        gt_chunks = row["contexts"]
        if isinstance(gt_chunks, str):
            gt_chunks = literal_eval(gt_chunks)

        # retrieve top-K chunks
        response = query_engine.query(question)
        retrieved_chunks = [node.get_content() for node in response.source_nodes[:k]]

        # Binarize relevance for each retrieved chunk
        relevance_list = [
            1 if is_relevant(chunk, gt_chunks, embed_model, threshold) else 0
            for chunk in retrieved_chunks
        ]

        # MRR
        rr = 0.0
        for rank, rel in enumerate(relevance_list, start=1):
            if rel == 1:
                rr = 1.0 / rank
                break

        # Recall@K
        recall = 1.0 if any(relevance_list) else 0.0

        results.append({
            "question": question,
            "contexts": row["contexts"],
            "document": row["document"],
            "retrieved_chunks": retrieved_chunks,
            "gt_chunks": gt_chunks,
            "relevance_list": relevance_list,
            "reciprocal_rank": rr,
            "recall": recall,
            "num_relevant": sum(relevance_list),
        })

    # --- Aggregate ---
    mrr = float(np.mean([r["reciprocal_rank"] for r in results]))
    recall_at_k = float(np.mean([r["recall"] for r in results]))

    print(f"Overall MRR@{k}:     {mrr:.4f}")
    print(f"Overall Recall@{k}:  {recall_at_k:.4f}")

    summary = {
        f"MRR@{k}": round(mrr, 4),
        f"Recall@{k}": round(recall_at_k, 4),
        "threshold_used": threshold,
        "num_questions": len(results),
        "per_question": results,
    }

    return summary

In [ ]:
results = evaluate_retrieval(
    query_engine=query_engine,
    eval_dataset=eval_dataset,
    embed_model=embed_model,
    threshold=0.80,
    k=5,
)

Overall MRR@5:     0.4286
Overall Recall@5:  0.4286


### Per-question analysis

In [21]:
per_q_df = pd.DataFrame(results["per_question"])
per_q_df.head()

,question,contexts,document,retrieved_chunks,gt_chunks,relevance_list,reciprocal_rank,recall,num_relevant
0,What is the order model number for the M12 cable?,[### 17-pin M12 Cable with Fast Ethernet Inter...,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,[| 8-pin terminal |\n\n17-pin M12 Ca...,[### 17-pin M12 Cable with Fast Ethernet Inter...,"[1, 0, 0, 0, 0]",1.0,1.0,1
1,What are the dimensions of the M12-mount housi...,[Fig. 1-1: $46 \times 57.6 \times 25$ mm Mecha...,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,[# 1 Mechanical Dimensions\n\nThe dimensions i...,[Fig. 1-1: $46 \times 57.6 \times 25$ mm Mecha...,"[1, 0, 0, 0, 0]",1.0,1.0,1
2,What are all the pins on the 17-pin connector ...,[| Purple White | 3 | - | - ...,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,[| Do not use it for wiring |\n| Purple | 13 ...,[| Purple White | 3 | - | - ...,"[0, 0, 0, 0, 0]",0.0,0.0,0
3,What communication protocols does the SM2 seri...,"[* Support Serial, TCP, UDP, FTP, Modbus and o...",User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,[Equipment shown in Fig. 1-2\n\nDouble-click t...,"[* Support Serial, TCP, UDP, FTP, Modbus and o...","[0, 0, 0, 0, 0]",0.0,0.0,0
4,List all the LED indicators and what each colo...,[| PWR Indicator | It is a power indicator. Th...,User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN,[# Status LED Description\n\n| Status LED |...,[| PWR Indicator | It is a power indicator. Th...,"[1, 0, 0, 0, 0]",1.0,1.0,1


### Failure cases

In [22]:
# Questions where nothing was retrieved correctly
failures = per_q_df[per_q_df["recall"] == 0.0]
print(f"\n{len(failures)} complete misses:\n")
for _, row in failures.iterrows():
    print(f"  [{row['document']}] {row['question']}")

# Questions where the first relevant chunk wasn't ranked #1
poor_mrr = per_q_df[(per_q_df["recall"] == 1.0) & (per_q_df["reciprocal_rank"] < 1.0)]
print(f"\n{len(poor_mrr)} retrieved but ranked poorly:\n")
for _, row in poor_mrr.iterrows():
    print(f"  RR={row['reciprocal_rank']:.2f} | {row['question']}")
    print(f"  Relevance list: {row['relevance_list']}")


4 complete misses:

  [User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN] What are all the pins on the 17-pin connector that should NOT be used for wiring?
  [User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN] What communication protocols does the SM2 series support?
  [User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN] Is the SM2 SE camera compatible with GigE Vision or GenICam standards?
  [User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN] Can the IO box V1.0 upgrade non-isolated IO to opto-isolated IO?

0 retrieved but ranked poorly:



In [23]:
def inspect_question(per_q_df, index):
    row = per_q_df.iloc[index]
    print(f"Question:     {row['question']}")
    print(f"Contexts: {row['contexts']}")
    print(f"RR: {row['reciprocal_rank']:.2f}  |  Recall: {row['recall']}")
    print(f"\nGT Chunks ({len(row['gt_chunks'])}):")
    for i, c in enumerate(row['gt_chunks']):
        print(f"  [{i}] {c[:200]}...")
    print(f"\nRetrieved Chunks (relevance: {row['relevance_list']}):")
    for i, (chunk, rel) in enumerate(zip(row['retrieved_chunks'], row['relevance_list'])):
        tag = "✓" if rel else "✗"
        print(f"  {tag} [{i}] {chunk[:200]}...")

inspect_question(per_q_df, 0)

Question:     What is the order model number for the M12 cable?
Contexts: ['### 17-pin M12 Cable with Fast Ethernet Interface\n\nOrder Model: VT-M1217P2RJ45-3M(SM2)\n\n']
RR: 1.00  |  Recall: 1.0

GT Chunks (1):
  [0] ### 17-pin M12 Cable with Fast Ethernet Interface

Order Model: VT-M1217P2RJ45-3M(SM2)

...

Retrieved Chunks (relevance: [1, 0, 0, 0, 0]):
  ✓ [0] | 8-pin terminal           |

17-pin M12 Cable with Fast Ethernet Interface 169.254.58.

Order Model: VT-M1217P2RJ45-3M(SM2)

Subnet Mask: 255.255.255.0

Default Gateway: 169.254.58.254

＊ The network...
  ✗ [1] I/O and serial ports

| Pin | Signal    | Signal Source | Designation                                                                    | Cable                    |
| --- | --------- | ------------- ...
  ✗ [2] | Do not use it for wiring |
| Green  | 4   | RS-232 TX | -             | RS-232 serial port output                                                      | DB9 female serial port   |
| Green  | 5   | R...
  ✗ [3

### Per-Document Analysis

In [24]:
doc_summary = per_q_df.groupby("document").agg(
    num_questions=("question", "count"),
    MRR=("reciprocal_rank", "mean"),
    Recall=("recall", "mean"),
).round(4)
print(doc_summary)

                                           num_questions     MRR  Recall
document                                                                
User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN              7  0.4286  0.4286


## Evaluate Generation

In [42]:
async def call_judge(system_prompt: str, user_message: str) -> dict:
    messages = [
        ChatMessage(role="system", content=system_prompt),
        ChatMessage(role="user", content=user_message),
    ]

    response = await llm.achat(messages)

    raw = response.message.content.strip()

    return raw

def format_contexts(contexts: list[str]) -> str:
    return "\n\n".join(f"[Chunk {i+1}]: {c}" for i, c in enumerate(contexts))

def analyze_metric(summary: dict, failure_threshold: float = 3.0) -> pd.DataFrame:
    """
    Returns per-question DataFrame and prints:
      - per-document aggregation
      - failure cases (score < failure_threshold)
    """
    metric = summary["metric"]
    df     = pd.DataFrame(summary["per_question"])

    # per-document
    print(f"\n{'='*60}")
    print(f"  {metric.upper()} — per-document summary")
    print(f"{'='*60}")
    doc_summary = (
        df.groupby("document")
          .agg(num_questions=("question", "count"), mean_score=("score", "mean"))
          .round(4)
    )
    print(doc_summary)

    # failures
    failures = df[df["score"] < failure_threshold]
    print(f"\n{len(failures)} failure(s) with score < {failure_threshold}:\n")
    for _, row in failures.iterrows():
        print(f"  [{row['document']}] score={row['score']} | {row['question']}")
        print(f"    Reasoning: {row['reasoning']}\n")

    return df


def inspect_generation_question(df: pd.DataFrame, index: int):
    """Detailed view of a single question result."""
    row = df.iloc[index]
    print(f"Metric:           {df.attrs.get('metric', 'N/A')}")
    print(f"Document:         {row['document']}")
    print(f"Question:         {row['question']}")
    if "ground_truth" in row:
        print(f"Ground Truth:     {row['ground_truth']}")
    print(f"Generated Answer: {row['generated_answer']}")
    if "retrieved_contexts" in row:
        print(f"\nRetrieved Contexts:")
        for i, c in enumerate(row["retrieved_contexts"]):
            print(f"  [{i+1}] {c[:300]}...")
    print(f"\nScore:     {row['score']}")
    print(f"Reasoning: {row['reasoning']}")

### Correctness Evaluation

In [ ]:
async def evaluate_correctness(
    eval_dataset: pd.DataFrame,
    generated_answers: list[str],
) -> dict:
    """
    Correctness: compare generated answer vs ground truth.
    Requires ground_truth column. Judge returns score 1-5.
    """
    results = []

    for (_, row), generated in zip(eval_dataset.iterrows(), generated_answers):
        user_msg = (
            f"Question: {row['question']}\n\n"
            f"Ground Truth Answer: {row['ground_truth']}\n\n"
            f"Generated Answer: {generated}"
        )
        verdict = await call_judge(correctness_prompt, user_msg)

        results.append({
            "document":         row["document"],
            "question":         row["question"],
            "ground_truth":     row["ground_truth"],
            "generated_answer": generated,
        })

    scores = [r["generated_answer"] for r in results]
    avg    = float(np.mean(scores))
    print(f"Correctness — mean score: {avg:.4f}  (scale 1–5)")

    return {
        "metric":        "correctness",
        "mean_score":    round(avg, 4),
        "num_questions": len(results),
        "per_question":  results,
    }

correctness_results  = await evaluate_correctness(eval_dataset, generated_answers)

INFO:google_genai.models:AFC is enabled with max remote calls: 10.
** Messages: **
system: system:
You are an expert evaluator system for a question answering system.
You need to evaluate the quality of the generated answer based on the question and reference ground truth answer.
You should output a single score between 1 to 5:
1 means the generated answer is irrelevant or incorrect, lacking coherence or relevance to the question and reference answer.
2 means the generated answer partially addresses the question and reference answer but contains significant inaccuracies or lacks clarity.
3 means the generated answer is generally accurate and relevant to the question and reference answer, but may contain minor errors or omissions.
4 means the generated answer demonstrates a high level of accuracy and relevance to the question and reference answer, with few errors or omissions.
5 means the generated answer is flawless, perfectly capturing the essence of the question and reference answer 

c:\Users\chamu\code\technical-manual-rag\.venv\Lib\site-packages\pygments\regexopt.py:89: RuntimeWarning: coroutine 'prepare_chat_params' was never awaited
  for group in groupby(strings, lambda s: s[0] == first[0])) \


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 31.424091767s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash-lite', 'location': 

In [45]:
correctness_df  = analyze_metric(correctness_results,  failure_threshold=3.0)

NameError: name 'correctness_results' is not defined

In [ ]:
inspect_generation_question(correctness_df,  index=0)

### Relevance Evaluation

In [ ]:
def evaluate_relevance(
    eval_dataset: pd.DataFrame,
    generated_answers: list[str],
) -> dict:
    """
    Answer Relevance: does the answer address the question?
    No ground truth needed. Judge returns score 1-5 + reasoning.
    """
    results = []

    for (_, row), generated in zip(eval_dataset.iterrows(), generated_answers):
        user_msg = (
            f"Question: {row['question']}\n\n"
            f"Generated Answer: {generated}"
        )
        verdict = call_judge(relevance_prompt, user_msg)

        results.append({
            "document":         row["document"],
            "question":         row["question"],
            "generated_answer": generated,
            "score":            verdict.get("score"),
            "reasoning":        verdict.get("reasoning"),
        })

    scores = [r["score"] for r in results]
    avg    = float(np.mean(scores))
    print(f"Relevance — mean score: {avg:.4f}  (scale 1–5)")

    return {
        "metric":        "relevance",
        "mean_score":    round(avg, 4),
        "num_questions": len(results),
        "per_question":  results,
    }

relevance_results    = evaluate_relevance(eval_dataset, generated_answers)

Retrieval Hit Rate: 1.00
INFO:google_genai.models:AFC is enabled with max remote calls: 10.
** Messages: **
system: You are an expert Q&A system that is trusted around the world.
Always answer the query using the provided context information, and not prior knowledge.
Some rules to follow:
1. Never directly reference the given context in your answer.
2. Avoid statements like 'Based on the context, ...' or 'The context information ...' or anything along those lines.
user: Context information is below.
---------------------
file_name: User_Manual_SM2_SE_Camera_CT_Ver.2.5.4_EN.pdf
page: 6

|
| Gray  | 6   | DI\_0     | It can be configured as input or output, and is input by default. |
| Black | 7   | GND       | DC Power Supply Negative                                          |
| Red   | 8   | POWER\_IN | DC Power Supply Positive                                          |

You cannot use DB9 female serial port and VCC to power the device at the same time. Otherwise, damaging to power sup

In [ ]:
relevance_df    = analyze_metric(relevance_results, failure_threshold=3.0)

In [ ]:
inspect_generation_question(relevance_df, index=0)

### Faithfulness Evaluation

In [ ]:
def evaluate_faithfulness(
    eval_dataset: pd.DataFrame,
    generated_answers: list[str],
    retrieved_contexts: list[list[str]],
) -> dict:
    """
    Faithfulness: are all claims in the answer grounded in the retrieved context?
    No ground truth needed. Judge returns score 1-5 + reasoning.
    """
    results = []

    for (_, row), generated, contexts in zip(
        eval_dataset.iterrows(), generated_answers, retrieved_contexts
    ):
        user_msg = (
            f"Question: {row['question']}\n\n"
            f"Retrieved Context:\n{format_contexts(contexts)}\n\n"
            f"Generated Answer: {generated}"
        )
        verdict = call_judge(faithfulness_prompt, user_msg)

        results.append({
            "document":          row["document"],
            "question":          row["question"],
            "generated_answer":  generated,
            "retrieved_contexts": contexts,
            "score":             verdict.get("score"),
            "reasoning":         verdict.get("reasoning"),
        })

    scores = [r["score"] for r in results]
    avg    = float(np.mean(scores))
    print(f"Faithfulness — mean score: {avg:.4f}  (scale 1–5)")

    return {
        "metric":        "faithfulness",
        "mean_score":    round(avg, 4),
        "num_questions": len(results),
        "per_question":  results,
    }

faithfulness_results = evaluate_faithfulness(eval_dataset, generated_answers, retrieved_contexts)

In [ ]:
faithfulness_df = analyze_metric(faithfulness_results, failure_threshold=3.0)

In [ ]:
inspect_generation_question(faithfulness_df, index=0)